# Card sorting sobre PRs aceptados despues de retrabajo

## Pregunta inicial

**Que tipos de problemas hacen que los pull requests generados por agentes de IA no sean aceptados de manera inmediata y requieran retrabajo antes de poder integrarse?**

Este notebook documenta el flujo usado para pasar desde el universo completo de PRs del dataset AIDev hasta una muestra revisable mediante card sorting. El foco actual son los PRs `merged_after_rework`: casos que finalmente fueron mergeados, pero solo despues de commits adicionales y comentarios humanos que permiten observar el retrabajo.

## Problema, motivacion y consecuencias

Medir solo si un PR fue mergeado no explica que ocurrio durante la revision. Un PR puede terminar aceptado y aun asi haber requerido correcciones, aclaraciones o ajustes sustantivos antes del merge.

Este flujo busca observar esa zona intermedia: contribuciones de agentes de IA que no fueron aceptadas inmediatamente, pero que si lograron integrarse despues de intervencion humana y commits adicionales.

## Enfoque metodologico: card sorting

El abordaje usa **card sorting abierto** para que las categorias emerjan desde las tarjetas, en lugar de imponer una taxonomia previa.

Pasos principales:

1. construir la poblacion `merged_after_rework`;
2. extraer una muestra estratificada por agente;
3. preparar tarjetas con evidencia textual;
4. clasificar manualmente los motivos de retrabajo;
5. analizar distribuciones por agente, lenguaje, tipo de tarea y complejidad.

## Embudo de datos

El embudo se calcula desde el resumen de muestreo generado por `sampling/stratified_sampler.py`. Los filtros poblacionales ocurren antes de estratificar.

## Universo bruto antes de construir la poblacion operacional

La tabla siguiente muestra los cortes principales antes de construir la muestra.

## Archivos usados

Las rutas vienen desde las constantes de los scripts de muestreo y preparacion. El notebook no mantiene rutas duplicadas salvo la busqueda minima de la raiz del repositorio.

In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "exploration" / "aidev").exists():
            return path
    raise RuntimeError("No se encontro la raiz del repositorio")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from exploration.aidev.notebook_flow import (
    build_agent_distribution,
    build_evidence_tables,
    build_files_table,
    build_funnel,
    build_outputs_flow,
    build_preparation_flow,
    build_raw_overview,
    build_template_preview,
    load_flow_artifacts,
    validate_flow,
)
from exploration.aidev.preparation.rejection_cards import MANUAL_TEMPLATE_FIELDS
from exploration.aidev.sampling.stratified_sampler import POPULATION_MODE, STRATA_FIELDS

artifacts = load_flow_artifacts(ROOT)
sampling_summary = artifacts.sampling_summary
preparation_summary = artifacts.preparation_summary
filter_counts = sampling_summary["population_filter_counts"]
sample_df = artifacts.sample_df
cards_df = artifacts.cards_df
template_df = artifacts.template_df

POPULATION_MODE, STRATA_FIELDS, MANUAL_TEMPLATE_FIELDS[-1], len(sample_df), len(cards_df)


('merged-after-rework', ['agent'], 'categoria_retrabajo_pre_merge', 300, 300)

## Paso 0: filtros poblacionales antes de estratificar

El corte clave es `commit_count > 1` y `human_comment_count > 0` sobre PRs ya mergeados. Esto evita mezclar rechazos definitivos con aceptaciones despues de retrabajo.

In [2]:
build_files_table(artifacts)


,artefacto,ruta
0,Resumen de muestreo,exploration/aidev/sampling/outputs/merged_afte...
1,Muestra estratificada,exploration/aidev/sampling/outputs/merged_afte...
2,Resumen de preparacion,exploration/aidev/preparation/outputs/merged_a...
3,Tarjetas con evidencia,exploration/aidev/preparation/outputs/merged_a...
4,Plantilla manual,exploration/aidev/preparation/outputs/merged_a...


In [3]:
build_raw_overview(sampling_summary)


,metrica,definicion,total
0,PRs totales en pull_request,Todos los registros del parquet pull_request,33596
1,PRs cerrados,state = closed,31284
2,PRs mergeados,merged_at no nulo,24014
3,PRs cerrados sin merge,state = closed y merged_at nulo,7270
4,PRs mergeados con commits adicionales,merged_at no nulo y commit_count > 1,6884
5,Poblacion operacional,"merged_at no nulo, commit_count > 1 y human_co...",3166


## Paso 1: estratificacion por agente

Una vez construida la poblacion operacional, se calcula una muestra estratificada de 300 PRs usando solo `agent` como variable de estratificacion.

In [4]:
build_funnel(artifacts)


,paso,criterio,total,retencion_vs_universo,retencion_vs_paso_anterior
0,Universo bruto AIDev,Todos los PRs en pull_request,33596,1.000000,1.000000
1,PRs mergeados,merged_at no nulo,24014,0.714787,0.714787
2,PRs mergeados con commits adicionales,commit_count > 1,6884,0.204905,0.286666
3,Poblacion antes de estratificar,commit_count > 1 y human_comment_count > 0,3166,0.094237,0.459907
4,Muestra estratificada por agente,cuotas proporcionales por agent,300,0.008930,0.094757
5,Tarjetas candidatas,una tarjeta por PR de la muestra,300,0.008930,1.000000
6,Tarjetas listas para card sorting,guardia de calidad human_comment_count > 0,300,0.008930,1.000000
7,Plantilla manual,archivo para categorizacion manual,300,0.008930,1.000000


### Distribucion por agente

La distribucion compara poblacion, cuotas de muestreo y tarjetas finales.

In [5]:
build_agent_distribution(artifacts)


,poblacion_antes_de_estratificar,muestra_estratificada,tarjetas_finales
Copilot,1526,145,145
Devin,907,86,86
OpenAI_Codex,480,45,45
Cursor,180,17,17
Claude_Code,73,7,7


## Paso 2: preparacion de tarjetas

Preparation recibe una muestra que ya fue filtrada por commits adicionales y comentarios humanos. El filtro `human_comment_count > 0` queda como guardia de calidad.

In [6]:
build_preparation_flow(artifacts)


,paso,criterio,filas
0,Tarjetas candidatas,PRs de la muestra antes de la guardia de calidad,300
1,Tarjetas listas,human_comment_count > 0,300
2,Descartes,sin comentarios humanos detectados en evidencia,0


### Evidencia disponible tras preparation

La preparacion prioriza reviews, comentarios inline, comentarios generales y timeline. La tabla resume la fuente principal seleccionada para cada tarjeta.

In [7]:
evidence, review_states = build_evidence_tables(artifacts)
display(evidence)
display(review_states)


,fuente_evidencia,tarjetas
0,pr_review_comment,160
1,pr_comment,89
2,pr_review,51


,estado_review,tarjetas
0,COMMENTED,256
2,APPROVED,25
1,CHANGES_REQUESTED,18
3,DISMISSED,1


## Paso 3: salida para card sorting

El flujo vigente produce tarjetas con evidencia y una plantilla manual para registrar categorias emergentes durante el card sorting.

In [8]:
build_outputs_flow(artifacts)


,artefacto,ruta,filas
0,Tarjetas con evidencia,exploration/aidev/preparation/outputs/merged_a...,300
1,Plantilla manual,exploration/aidev/preparation/outputs/merged_a...,300


In [9]:
build_template_preview(artifacts)


,card_id,pr_id,pr_state,merged,repo_id,html_url,categoria_retrabajo_pre_merge
0,3078006902-A,3078006902,closed,True,354647574,https://github.com/spacelift-io/spacectl/pull/324,NaN
1,3154662508-A,3154662508,closed,True,737898780,https://github.com/maybe-finance/maybe/pull/2389,NaN
2,3184725856-A,3184725856,closed,True,357728969,https://github.com/oven-sh/bun/pull/20698,NaN
3,3193198936-A,3193198936,closed,True,707764474,https://github.com/tphakala/birdnet-go/pull/841,NaN
4,3224085262-A,3224085262,closed,True,753589979,https://github.com/karakeep-app/karakeep/pull/...,NaN


## Validaciones de consistencia

Estas validaciones hacen explicitos los supuestos del flujo: poblacion `merged_after_rework`, estratificacion por agente, muestra de 300 PRs y una tarjeta final por PR.

In [10]:
validate_flow(artifacts)


'Validaciones completadas'

## Lectura metodologica

El flujo parte del universo completo de PRs, filtra antes del muestreo los casos que no muestran retrabajo observable y conserva una muestra estratificada por agente. La interpretacion cualitativa debe enfocarse en motivos de retrabajo antes del merge, no en rechazo definitivo.